The data used to prepare this indicator can be found here https://tims.berkeley.edu/summary.php

Need to manually download data for the most recent 10 years under the Pedestrian and Bicyclist filters for both Fatalities and Fatalities/Serious Injuries.  Then, rename and save those workbooks according to the data already found in here https://sacog.sharepoint.com/sites/RegionalMonitoringandReporting/Shared%20Documents/Forms/AllItems.aspx?id=%2Fsites%2FRegionalMonitoringandReporting%2FShared%20Documents%2FData%2FSafe%20Equitable%20Resilient%20Infrastructure%2FSafety%2FTIMS%2FBicyclists%20vs%20Pedestrians&viewid=8fba2a3f%2Dea8a%2D4663%2Da31f%2Dc1760cb6922d  

***

Preparing Workspace

***

In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import xlwt
from xlwt.Workbook import *
from pandas import ExcelWriter
import xlsxwriter
import time
import functools as ft
# pd.options.display.float_format = '{:.0f}'.format

# Plotting
import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

## Set file paths
if user == 'jfontes':
    # Git
    path_git     = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')

    # SharePoint
    path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
    path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'TIMS Data')
    path_main = os.path.join(path_sp, 'Data')
    path_out  = os.path.join(path_main, 'Safe Equitable Resilient Infrastructure', 'Safety')
    path_tims = os.path.join(path_out, 'TIMS')

    
path_code    = os.path.join(path_git, 'Data', 'TIMS')
path_config0 = os.path.join(path_git , 'config')
path_config  = os.path.join(path_code, 'config')


In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

***

Importing

***

In [ ]:
path_in = os.path.join(path_tims, 'Bicyclists vs Pedestrians', 'Fatalities')
df_counties1_f = pd.read_csv(os.path.join(path_in, 'Counties_Fatalities_Bicyclists.csv' ))
df_counties2_f = pd.read_csv(os.path.join(path_in, 'Counties_Fatalities_Pedestrians.csv'))

df_jurisdictions1_f = pd.read_csv(os.path.join(path_in, 'Jurisdictions_Fatalities_Bicyclists.csv' ))
df_jurisdictions2_f = pd.read_csv(os.path.join(path_in, 'Jurisdictions_Fatalities_Pedestrians.csv'))

path_in = os.path.join(path_tims, 'Bicyclists vs Pedestrians', 'Serious Injuries')
df_counties1_i = pd.read_csv(os.path.join(path_in, 'Counties_Fatalities_Serious_Injuries_Bicyclists.csv' ))
df_counties2_i = pd.read_csv(os.path.join(path_in, 'Counties_Fatalities_Serious_Injuries_Pedestrians.csv'))

df_jurisdictions1_i = pd.read_csv(os.path.join(path_in, 'Jurisdictions_Fatalities_Serious_Injuries_Bicyclists.csv' ))
df_jurisdictions2_i = pd.read_csv(os.path.join(path_in, 'Jurisdictions_Fatalities_Serious_Injuries_Pedestrians.csv'))

df_counties1_f     ['Case'] = 'Fatalities'
df_counties2_f     ['Case'] = 'Fatalities'
df_jurisdictions1_f['Case'] = 'Fatalities'
df_jurisdictions2_f['Case'] = 'Fatalities'

df_counties1_i     ['Case'] = 'Fatalities and Serious Injuries'
df_counties2_i     ['Case'] = 'Fatalities and Serious Injuries'
df_jurisdictions1_i['Case'] = 'Fatalities and Serious Injuries'
df_jurisdictions2_i['Case'] = 'Fatalities and Serious Injuries'

df_counties1_f      = df_counties1_f     .replace(',', '', regex=True)
df_counties2_f      = df_counties2_f     .replace(',', '', regex=True)
df_jurisdictions1_f = df_jurisdictions1_f.replace(',', '', regex=True)
df_jurisdictions2_f = df_jurisdictions2_f.replace(',', '', regex=True)

df_counties1_i      = df_counties1_i     .replace(',', '', regex=True)
df_counties2_i      = df_counties2_i     .replace(',', '', regex=True)
df_jurisdictions1_i = df_jurisdictions1_i.replace(',', '', regex=True)
df_jurisdictions2_i = df_jurisdictions2_i.replace(',', '', regex=True)

df_counties1      = pd.concat([df_counties1_f     , df_counties1_i     ])
df_counties2      = pd.concat([df_counties2_f     , df_counties2_i     ])
df_jurisdictions1 = pd.concat([df_jurisdictions1_f, df_jurisdictions1_i])
df_jurisdictions2 = pd.concat([df_jurisdictions2_f, df_jurisdictions2_i])

df_counties1      = df_counties1     .drop('Average', axis=1)
df_counties2      = df_counties2     .drop('Average', axis=1)
df_jurisdictions1 = df_jurisdictions1.drop('Average', axis=1)
df_jurisdictions2 = df_jurisdictions2.drop('Average', axis=1)

df_jurisdictions1 = df_jurisdictions1[df_jurisdictions1['County'] != 'Statewide']
df_jurisdictions2 = df_jurisdictions2[df_jurisdictions2['County'] != 'Statewide']

df_counties1      = pd.melt(df_counties1     , id_vars = ['County'        , 'Case'], var_name = 'Year', value_name = 'Count')
df_counties2      = pd.melt(df_counties2     , id_vars = ['County'        , 'Case'], var_name = 'Year', value_name = 'Count')
df_jurisdictions1 = pd.melt(df_jurisdictions1, id_vars = ['County', 'City', 'Case'], var_name = 'Year', value_name = 'Count')
df_jurisdictions2 = pd.melt(df_jurisdictions2, id_vars = ['County', 'City', 'Case'], var_name = 'Year', value_name = 'Count')

df_counties1     ['Count'] = df_counties1     ['Count'].astype('float32')
df_counties2     ['Count'] = df_counties2     ['Count'].astype('float32')
df_jurisdictions1['Count'] = df_jurisdictions1['Count'].astype('float32')
df_jurisdictions2['Count'] = df_jurisdictions2['Count'].astype('float32')

df_counties1      = df_counties1     .pivot_table(index = ['County',         'Year'], columns = 'Case', values = 'Count').reset_index()
df_counties2      = df_counties2     .pivot_table(index = ['County',         'Year'], columns = 'Case', values = 'Count').reset_index()
df_jurisdictions1 = df_jurisdictions1.pivot_table(index = ['County', 'City', 'Year'], columns = 'Case', values = 'Count').reset_index()
df_jurisdictions2 = df_jurisdictions2.pivot_table(index = ['County', 'City', 'Year'], columns = 'Case', values = 'Count').reset_index()

df_counties1     ['Serious Injuries'] = df_counties1     ['Fatalities and Serious Injuries'] - df_counties1     ['Fatalities']
df_counties2     ['Serious Injuries'] = df_counties2     ['Fatalities and Serious Injuries'] - df_counties2     ['Fatalities']
df_jurisdictions1['Serious Injuries'] = df_jurisdictions1['Fatalities and Serious Injuries'] - df_jurisdictions1['Fatalities']
df_jurisdictions2['Serious Injuries'] = df_jurisdictions2['Fatalities and Serious Injuries'] - df_jurisdictions2['Fatalities']

df_counties1      = df_counties1     .drop(['Fatalities and Serious Injuries'], axis=1)
df_counties2      = df_counties2     .drop(['Fatalities and Serious Injuries'], axis=1)
df_jurisdictions1 = df_jurisdictions1.drop(['Fatalities and Serious Injuries'], axis=1)
df_jurisdictions2 = df_jurisdictions2.drop(['Fatalities and Serious Injuries'], axis=1)

df_counties1     ['Category'] = 'Bicyclists'
df_counties2     ['Category'] = 'Pedestrians'
df_jurisdictions1['Category'] = 'Bicyclists'
df_jurisdictions2['Category'] = 'Pedestrians'

df_counties      = pd.concat([df_counties1     , df_counties2     ])
df_jurisdictions = pd.concat([df_jurisdictions1, df_jurisdictions2])

df_counties     ['County'] = df_counties     ['County'].str.title()
df_jurisdictions['County'] = df_jurisdictions['County'].str.title()
df_jurisdictions['City'  ] = df_jurisdictions['City'  ].str.title()

df_counties      = df_counties     .sort_values(['County',         'Year', 'Category'], ascending = [True,       True, True])
df_jurisdictions = df_jurisdictions.sort_values(['County', 'City', 'Year', 'Category'], ascending = [True, True, True, True])


df_state    = df_counties[df_counties['County'] == 'Statewide'].rename(columns = {'County':'Geography'})
df_counties = df_counties[df_counties['County'] != 'Statewide']

df_mpo = df_counties[df_counties['County'].isin(['El Dorado', 'Placer', 'Sacramento', 'Sutter', 'Yolo', 'Yuba'])]
df_mpo = df_mpo.drop(['County'], axis=1)
df_mpo = df_mpo.groupby(['Year', 'Category'], as_index=False).sum()
df_mpo['MPO'] = 'SACOG'

df_jurisdictions['Year'] = df_jurisdictions['Year'].astype(int)
df_counties     ['Year'] = df_counties     ['Year'].astype(int)
df_mpo          ['Year'] = df_mpo          ['Year'].astype(int)
df_state        ['Year'] = df_state        ['Year'].astype(int)

df_jurisdictions['Fatalities_5 Year Average'      ] = df_jurisdictions.groupby(['County', 'City', 'Category'])['Fatalities'      ].transform(lambda x: x.rolling(5, 1).mean())
df_jurisdictions['Serious Injuries_5 Year Average'] = df_jurisdictions.groupby(['County', 'City', 'Category'])['Serious Injuries'].transform(lambda x: x.rolling(5, 1).mean())
df_jurisdictions.loc[df_jurisdictions['Year'].isin([2014, 2015, 2016, 2017]), 'Fatalities_5 Year Average'      ] = np.nan
df_jurisdictions.loc[df_jurisdictions['Year'].isin([2014, 2015, 2016, 2017]), 'Serious Injuries_5 Year Average'] = np.nan

df_counties['Fatalities_5 Year Average'      ] = df_counties.groupby(['County', 'Category'])['Fatalities'      ].transform(lambda x: x.rolling(5, 1).mean())
df_counties['Serious Injuries_5 Year Average'] = df_counties.groupby(['County', 'Category'])['Serious Injuries'].transform(lambda x: x.rolling(5, 1).mean())
df_counties.loc[df_counties['Year'].isin([2014, 2015, 2016, 2017]), 'Fatalities_5 Year Average'      ] = np.nan
df_counties.loc[df_counties['Year'].isin([2014, 2015, 2016, 2017]), 'Serious Injuries_5 Year Average'] = np.nan

df_mpo['Fatalities_5 Year Average'      ] = df_mpo.groupby(['MPO', 'Category'])['Fatalities'      ].transform(lambda x: x.rolling(5, 1).mean())
df_mpo['Serious Injuries_5 Year Average'] = df_mpo.groupby(['MPO', 'Category'])['Serious Injuries'].transform(lambda x: x.rolling(5, 1).mean())
df_mpo.loc[df_mpo['Year'].isin([2014, 2015, 2016, 2017]), 'Fatalities_5 Year Average'      ] = np.nan
df_mpo.loc[df_mpo['Year'].isin([2014, 2015, 2016, 2017]), 'Serious Injuries_5 Year Average'] = np.nan

df_state['Fatalities_5 Year Average'      ] = df_state.groupby(['Geography', 'Category'])['Fatalities'      ].transform(lambda x: x.rolling(5, 1).mean())
df_state['Serious Injuries_5 Year Average'] = df_state.groupby(['Geography', 'Category'])['Serious Injuries'].transform(lambda x: x.rolling(5, 1).mean())
df_state.loc[df_state['Year'].isin([2014, 2015, 2016, 2017]), 'Fatalities_5 Year Average'      ] = np.nan
df_state.loc[df_state['Year'].isin([2014, 2015, 2016, 2017]), 'Serious Injuries_5 Year Average'] = np.nan


df_jurisdictions = df_jurisdictions.sort_values(['County'   , 'City', 'Year', 'Category'], ascending = [True, True, False, True])
df_counties      = df_counties     .sort_values(['County'   ,         'Year', 'Category'], ascending = [True,       False, True])
df_mpo           = df_mpo          .sort_values(['MPO'      ,         'Year', 'Category'], ascending = [True,       False, True])
df_state         = df_state        .sort_values(['Geography',         'Year', 'Category'], ascending = [True,       False, True])


df_jurisdictions = df_jurisdictions.set_index(['County'   , 'City', 'Year', 'Category']).reset_index()
df_counties      = df_counties     .set_index(['County'   ,         'Year', 'Category']).reset_index()
df_mpo           = df_mpo          .set_index('MPO'                                    ).reset_index()
df_state         = df_state        .set_index(['Geography',         'Year', 'Category']).reset_index()

df_jurisdictions = df_jurisdictions.rename(columns = {'City':'Jurisdiction'})


display(df_jurisdictions.head(3)); print('')
display(df_counties     .head(3)); print('')
display(df_mpo          .head(3)); print('')
display(df_state        .head(3))

***

Exporting

***

In [ ]:
# # Update overall about documentations workbook
# sample_type = 'TIMS'
# indicator_name = 'Safety_4'
# year_start = df_counties['Year'].min()
# year_end   = df_counties['Year'].max()

# df_about = write_about(sample_type      = sample_type
#                        , indicator_name = indicator_name
#                        , year_start     = year_start
#                        , year_end       = year_end
#                        , path_config0   = path_config0
#                        )
# with pd.ExcelWriter(os.path.join(path_main, 'About Indicators.xlsx'), mode = 'a', engine = 'openpyxl', if_sheet_exists = 'replace') as writer:
#     df_about.to_excel(writer, index = False, sheet_name = indicator_name, header = False)


In [ ]:
sample_type = 'TIMS'
indicator_name = 'Safety_4'
year_start = df_counties['Year'].min()
year_end   = df_counties['Year'].max()

path_out  = os.path.join(path_main, 'Safe Equitable Resilient Infrastructure', 'Safety', 'Safety_4 Bicyclists vs Pedestrians')

geography = 'Jurisdictions'
df_about = write_about(sample_type      = sample_type
                           , indicator_name = indicator_name
                           , year_start     = year_start
                           , year_end       = year_end
                           , path_config0   = path_config0
                           , geography      = geography)
with pd.ExcelWriter(os.path.join(path_out, f'{indicator_name} {geography} {sample_type}.xlsx'), engine='xlsxwriter') as writer:
    df_about        .to_excel(writer, index = False, sheet_name = 'About'        , header = False)
    df_jurisdictions.to_excel(writer, index = False, sheet_name = geography                      )
geography = 'Counties'
df_about = write_about(sample_type      = sample_type
                           , indicator_name = indicator_name
                           , year_start     = year_start
                           , year_end       = year_end
                           , path_config0   = path_config0
                           , geography      = geography)
with pd.ExcelWriter(os.path.join(path_out, f'{indicator_name} {geography} {sample_type}.xlsx'), engine='xlsxwriter') as writer:
    df_about   .to_excel(writer, index = False, sheet_name = 'About'   , header = False)
    df_counties.to_excel(writer, index = False, sheet_name = geography                 )
geography = 'MPO'
df_about = write_about(sample_type      = sample_type
                           , indicator_name = indicator_name
                           , year_start     = year_start
                           , year_end       = year_end
                           , path_config0   = path_config0
                           , geography      = geography)
with pd.ExcelWriter(os.path.join(path_out, f'{indicator_name} {geography} {sample_type}.xlsx'), engine='xlsxwriter') as writer:
    df_about.to_excel(writer, index = False, sheet_name = 'About', header = False)
    df_mpo  .to_excel(writer, index = False, sheet_name = geography              )
geography = 'Statewide'
df_about = write_about(sample_type      = sample_type
                           , indicator_name = indicator_name
                           , year_start     = year_start
                           , year_end       = year_end
                           , path_config0   = path_config0
                           , geography      = geography)
with pd.ExcelWriter(os.path.join(path_out, f'{indicator_name} {geography} {sample_type}.xlsx'), engine='xlsxwriter') as writer:
    df_about.to_excel(writer, index = False, sheet_name = 'About'    , header = False)
    df_state.to_excel(writer, index = False, sheet_name = geography                  )

In [ ]:
path_out = r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring\Data"

geography = 'Jurisdictions'
df_about = write_about(sample_type      = sample_type
                       , indicator_name = indicator_name
                       , year_start     = year_start
                       , year_end       = year_end
                       , path_config0   = path_config0
                       , geography      = geography)
with pd.ExcelWriter(os.path.join(path_out, f'{indicator_name} {geography} {sample_type}.xlsx'), engine='xlsxwriter') as writer:
    df_about        .to_excel(writer, index = False, sheet_name = 'About'        , header = False)
    df_jurisdictions.to_excel(writer, index = False, sheet_name = geography                      )
geography = 'Counties'
df_about = write_about(sample_type      = sample_type
                       , indicator_name = indicator_name
                       , year_start     = year_start
                       , year_end       = year_end
                       , path_config0   = path_config0
                       , geography      = geography)
with pd.ExcelWriter(os.path.join(path_out, f'{indicator_name} {geography} {sample_type}.xlsx'), engine='xlsxwriter') as writer:
    df_about   .to_excel(writer, index = False, sheet_name = 'About'   , header = False)
    df_counties.to_excel(writer, index = False, sheet_name = geography                 )
geography = 'MPO'
df_about = write_about(sample_type      = sample_type
                       , indicator_name = indicator_name
                       , year_start     = year_start
                       , year_end       = year_end
                       , path_config0   = path_config0
                       , geography      = geography)
with pd.ExcelWriter(os.path.join(path_out, f'{indicator_name} {geography} {sample_type}.xlsx'), engine='xlsxwriter') as writer:
    df_about.to_excel(writer, index = False, sheet_name = 'About', header = False)
    df_mpo  .to_excel(writer, index = False, sheet_name = geography              )
geography = 'Statewide'
df_about = write_about(sample_type      = sample_type
                       , indicator_name = indicator_name
                       , year_start     = year_start
                       , year_end       = year_end
                       , path_config0   = path_config0
                       , geography      = geography)
with pd.ExcelWriter(os.path.join(path_out, f'{indicator_name} {geography} {sample_type}.xlsx'), engine='xlsxwriter') as writer:
    df_about.to_excel(writer, index = False, sheet_name = 'About'    , header = False)
    df_state.to_excel(writer, index = False, sheet_name = geography                  )

In [ ]:
# # Set Indicator
# indicator_name = 'Safety_4'
# plot_name = 'fatality_bikes_pedestrians'
# export = False


# ## Importing ---

# path_in = os.path.join(path_tims, 'Bikes vs Pedestrians', 'Fatalities')
# df_counties1 = pd.read_csv(os.path.join(path_in, 'Counties_Fatalities_Bikes.csv'      ))
# df_counties2 = pd.read_csv(os.path.join(path_in, 'Counties_Fatalities_Pedestrians.csv'))


# ## Organizing ---


# df_counties1 = df_counties1.drop('Average', axis=1)
# df_counties2 = df_counties2.drop('Average', axis=1)

# df_counties1 = pd.melt(df_counties1, id_vars = ['County'], var_name = 'Year', value_name = 'Fatalities')
# df_counties2 = pd.melt(df_counties2, id_vars = ['County'], var_name = 'Year', value_name = 'Fatalities')

# df_counties1     ['Category'] = 'Bikes'
# df_counties2     ['Category'] = 'Pedestrians'

# df_plot = pd.concat([df_counties1, df_counties2])


# df_plot['County'] = df_plot['County'].str.title()

# df_plot = df_plot[df_plot['County'].isin(['El Dorado', 'Sacramento', 'Placer', 'Sutter', 'Yolo', 'Yuba'])]

# df_plot['Year'] = df_plot['Year'].astype(int)
# df_plot['Fatalities'] = df_plot['Fatalities'].astype(int)

# df_plot = df_plot.groupby(['Year', 'Category'], as_index=False)['Fatalities'].sum()
# df_plot = df_plot.sort_values(['Year', 'Category'], ascending = [False, False])

# df_plot = df_plot.set_index(['Year', 'Category']).reset_index()

# display(df_plot.head())


# ## Plotting ---


# color_map = {
#     'Bikes': "#9DC209"
#     , 'Pedestrians': "#1E90FF"
# }


# fig = px.bar(df_plot, y='Fatalities', x='Year'
#              , color='Category'
#              , color_discrete_map=color_map)


# title = 'Fatalities by Counties, Bikes vs Pedestrians'
# # fig.update_layout(showlegend = False)
# fig.update_xaxes(tick0=0, dtick=1)
# fig.update_traces(hovertemplate='%{y}')


# plot_agol(export=export)